# 04 -- Luck Score Demo

**Contact Luck Prototype v0.1**

Walks through the full scoring pipeline for a single play: predicted outcome distribution -> expected ordinal value -> preliminary additive raw luck -> provisional -100..+100 public score -> data-completeness confidence report.

If no real cleaned data / trained artifact is available, this notebook falls back to a small synthetic example so the pipeline is always runnable.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.

In [ ]:
import pandas as pd

from mlb_luck_score.config import CLASS_ORDER, DEFAULT_VALUE_MAP, PROCESSED_DATA_DIR
from mlb_luck_score.models.predict_outcomes import _build_synthetic_demo_data
from mlb_luck_score.models.train_contact_model import predict_proba_ordered, train_model
from mlb_luck_score.scoring.confidence import compute_confidence
from mlb_luck_score.scoring.public_score import raw_luck_to_public_score
from mlb_luck_score.scoring.raw_luck import compute_expected_value, compute_raw_luck

CLEANED_PATH = PROCESSED_DATA_DIR / "cleaned_batted_balls.parquet"
pd.set_option("display.width", 120)

In [ ]:
if CLEANED_PATH.exists():
    df = pd.read_parquet(CLEANED_PATH)
    training_eligible = df[df["eligible_for_training"].astype(bool)]
    if len(training_eligible) >= 10:
        source_df = training_eligible
        print(f"Using real cleaned data: {len(source_df)} training-eligible rows")
    else:
        source_df = _build_synthetic_demo_data()
        print("Real cleaned data has too few training-eligible rows; using synthetic demo data instead.")
else:
    source_df = _build_synthetic_demo_data()
    print(
        f"No cleaned data found at {CLEANED_PATH}; using a small synthetic demo dataset instead.\n"
        "Run `make download-sample` then `make clean-data` for a real-data walkthrough."
    )

## Train a quick model and pick one example play

In [ ]:
trained = train_model(source_df)
example = source_df.iloc[[0]].copy()
observed_outcome = str(example["outcome_class"].iloc[0])
feature_cols = trained.numeric_features + trained.categorical_features
proba_df = predict_proba_ordered(trained, example[feature_cols])
probabilities = {cls: float(proba_df.iloc[0][cls]) for cls in CLASS_ORDER}
example[list(CLASS_ORDER)] = [probabilities[c] for c in CLASS_ORDER]
display(example)

## Scoring walkthrough

In [ ]:
expected_value = compute_expected_value(probabilities, value_map=DEFAULT_VALUE_MAP)
actual_value = DEFAULT_VALUE_MAP[observed_outcome]
raw_luck = compute_raw_luck(probabilities, observed_outcome, value_map=DEFAULT_VALUE_MAP)
public_score = raw_luck_to_public_score(raw_luck)
confidence = compute_confidence(example.iloc[0])

print(f"Observed outcome:              {observed_outcome}")
print("Predicted probability distribution:")
for cls in CLASS_ORDER:
    print(f"  {cls:>9s}: {probabilities[cls]:.3f}")
print(f"P(observed result):             {probabilities[observed_outcome]:.3f}")
print(f"Expected ordinal value:         {expected_value:.3f}")
print(f"Actual ordinal value:           {actual_value:.3f}")
print(f"Preliminary raw luck (additive):{raw_luck:+.3f}")
print(f"Preliminary public score:       {public_score:+.1f} / 100 (NOT additive)")
print(f"Data-completeness label:        {confidence.label}")
print(f"Data-completeness detail:       {confidence}")

## Explicit provisional disclaimer

- The ordinal value map (`out=0, single=1, double=2, triple=3, home_run=4`) is a **Version 0.1 research placeholder**, not a validated run-value model.
- The public score's `tanh`-based mapping to [-100, 100] is a **provisional convenience choice**, not an empirically calibrated scientific mapping. It has not been fit against any historical distribution of raw-luck values.
- The confidence label reflects **observable data completeness only** -- it is not a statistical uncertainty estimate, and it does not affect (and should never affect) the raw-luck or public-score values above.
- Contact luck as computed here does **not** separate out decision quality, defensive execution, weather, or park effects -- see README.md `Future work`.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.